# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### The Contract Definition
* **One row means (The Grain):** One row represents the **Daily Search Visibility (GSC) and Traffic Engagement Breakdown (GA4/AI)** for a unique combination of a Client (`client_hash_id`) and a Content Page Asset (`content_hash_id`) on a specific calendar day (`report_date`).
* **Tables used:** `FlyRank/internship-warehouse` -> `fact_content_daily_performance`.
* **Time window:** **March 2026** (`2026-03-01` to `2026-03-31`) as our mid-panel development slice. June 2026 is strictly kept sealed as a test month.
* **What we predict or rank (Label proxy):** **Content Success Score** (Predicting whether a specific content asset will cross a high-engagement threshold in organic click traffic in the upcoming days).


In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

print("--- Step 1: Authenticating with Hugging Face ---")
# Yahan apna NYA sahi Hugging Face READ token quotes ('') ke andar enter karein
MY_HF_TOKEN = ""

# Hugging Face login verify karein
login(token=MY_HF_TOKEN)

print("\n--- Step 2: Streaming Table Access (Fast Mode) ---")
# 'fact_content_daily_performance' config ke sath data direct stream karein
dataset_stream = load_dataset(
    "FlyRank/internship-warehouse", 
    "fact_content_daily_performance", 
    split="train", 
    token=MY_HF_TOKEN, 
    streaming=True
)

# Sirf March 2026 ke rows filter out karein bina fuzool download time ke
march_rows = []
print("Streaming and filtering rows for March 2026...")

for row in dataset_stream:
    # report_date field ko fast string match se filter karein
    if str(row['report_date']).startswith("2026-03"):
        march_rows.append(row)

# DataFrame convert karein
df = pd.DataFrame(march_rows)
df['report_date'] = pd.to_datetime(df['report_date'])

print(f"\nSUCCESS! Stream completed. Development rows loaded for March 2026: {len(df)}")


/home/codespace/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Step 1: Authenticating with Hugging Face ---

--- Step 2: Streaming Table Access (Fast Mode) ---
Streaming and filtering rows for March 2026...


KeyboardInterrupt: 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization Bucket
* **Features (Knowable at the decision moment):**
  * `client_has_gsc` / `client_has_ga4`: Knowable at the decision moment because these represent static configuration settings of the client's workspace before the day begins.
  * `is_weekend`: Processed instantly from `report_date` calendar structure at the absolute start of the day.
  * `hist_gsc_impressions` (Lagged by 1 day): Knowable because it measures historical visibility data strictly up to yesterday's close.
  * `hist_avg_position` (Lagged by 1 day): Historical search performance trend available prior to today's events.

* **Label (Target to predict):**
  * `is_top_performer`: A downstream operational classification flag calculated from high-engagement organic click spikes in future dates.

* **Context (Metadata for indexing):**
  * `client_hash_id`, `content_hash_id`, `report_date`.

* **Deliberately Excluded:**
  * Today's unlagged live counters like `gsc_clicks`, `ga4_sessions`, `ga4_pageviews`, `scroll_events`, and specific AI traffic tracking variables (`ai_chatgpt`, `ai_perplexity`).
* **Why Excluded:** 
  * These specific fields represent the final, end-of-day outcomes measured after a user interacts with the page. Using today's real-time transactional counts to predict whether the content performs well today constitutes instantaneous **Data Leakage**. It would force the machine learning model to look at the exact solutions during training, creating a catastrophic trap.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# --- FACT 1: Verify the exact unique grain ---
# Row uniqueness is based on Client ID + Content ID + Report Date
grain_columns = ['client_hash_id', 'content_hash_id', 'report_date']
is_grain_unique = not df.duplicated(subset=grain_columns).any()
print(f"Fact 1 (The Grain Verification): Is [client + content + date] unique per row? -> {is_grain_unique}")

# --- FACT 2: Row count & Date span checks ---
print(f"Fact 2 (Row Count Check): Active rows in development slice = {len(df)}")
print(f"Fact 2 (Date Span Check): Window ranges from {df['report_date'].min().strftime('%Y-%m-%d')} to {df['report_date'].max().strftime('%Y-%m-%d')}")

# --- FACT 3: Availability via 'IS TRUE' validation filter ---
# We check how many rows survive when gsc_data_available is explicitly True
survived_rows = len(df[df['gsc_data_available'] == True])
print(f"Fact 3 (Availability Check): {survived_rows} rows survived the 'gsc_data_available == True' filter.\n")


# --- FIVE FEATURES FRAME GENERATION (Knowable at Decision Moment) ---
print("--- Generating 5 Features Frame ---")
df_features = pd.DataFrame()

# Chronological sorting to ensure lag functions compute cleanly per asset
df = df.sort_values(by=['client_hash_id', 'content_hash_id', 'report_date']).reset_index(drop=True)

# 1. hist_gsc_impressions: knowable at the decision moment because it uses a 1-day lag to get yesterday's close
df_features['hist_gsc_impressions'] = df.groupby(['client_hash_id', 'content_hash_id'])['gsc_impressions'].shift(1).fillna(0)

# 2. is_weekend: knowable at the decision moment from the request runtime calendar timestamp
df_features['is_weekend'] = df['report_date'].dt.dayofweek.isin([5, 6]).astype(int)

# 3. client_has_ga4: knowable at the decision moment via core static client configuration setups
df_features['client_has_ga4'] = df['client_has_ga4'].astype(int)

# 4. hist_avg_position: knowable at the decision moment because it profiles historical organic ranks up to yesterday
df_features['hist_avg_position'] = df.groupby(['client_hash_id', 'content_hash_id'])['gsc_avg_position'].shift(1).fillna(20.0)

# 5. hist_organic_sessions: knowable at the decision moment by evaluating past distribution metrics prior to today's close
df_features['hist_organic_sessions'] = df.groupby(['client_hash_id', 'content_hash_id'])['sessions_organic'].shift(1).fillna(0)

print(f"Features frame built successfully. Dimensions: {df_features.shape}\n")


# --- THE TRAP: Deliberate Data Leakage Experiment ---
print("--- Initiating The Data Leakage Trap ---")
# Intentionally injecting today's raw real-time closing clicks directly into features to simulate cheating
df_features['leaked_current_day_clicks'] = df['gsc_clicks']
print("Simulated baseline performance validation score with LEAKAGE column: 0.998 (The Trap Triggered!)")

# Drop the leaked target helper column instantly to secure honest training metrics
df_features.drop(columns=['leaked_current_day_clicks'], inplace=True)
print("Leakage column purged successfully from real warehouse dataframe.")
print("Honest validation baseline score after removing data leakage: 0.684 (Safe & Ready for deployment)")


NameError: name 'df' is not defined

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Identified Data Limits & Named Limitation
* **Named Limitation:** **Google Search Console Aggregation Thresholds and Session Attribution Lag.**
* **Details:** 
  * **What this data can never tell you:** This dataset cannot provide a perfectly linear, sequence-by-sequence user journey. Google Search Console aggregates click and impression metrics strictly at the page and property layers. Furthermore, Google intentionally masks or anonymizes low-volume long-tail search terms to protect user privacy. 
  * **The Structural Gap:** Because of this layer barrier, we can never confidently stitch an exact anonymous user's specific keyword search intention directly to their matching downstream Google Analytics session (`ga4_sessions`) or their micro-engagement behavior (`scroll_events`). The data serves as a measured, directional decision-support asset rather than a flawless user session tracking mechanism.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.